# Filtros de suavizado y reducción de ruido

En este cuaderno vas a comparar distintos filtros de suavizado sobre imágenes con ruido controlado. La idea central es que no todos los ruidos se corrigen igual ni todos los filtros preservan el detalle de la misma manera.


## Objetivo

Entender qué tipo de ruido intenta resolver cada filtro y qué costo visual tiene esa corrección.

## Resultados de aprendizaje

Al finalizar este cuaderno, vas a poder:

- distinguir ruido gaussiano y ruido sal y pimienta;
- aplicar filtros promedio, gaussiano, mediana y bilateral;
- comparar cuál funciona mejor según el tipo de degradación;
- describir la relación entre suavizado y pérdida de detalle.

## Relación con la secuencia

Este cuaderno se ubica después de mejora de imagen y antes de operaciones morfológicas e inpainting. La idea es trabajar primero con filtros que modifican intensidades vecinas y después pasar a limpiezas más específicas.


## Módulos que vamos a usar

- `cv2`: para aplicar filtros de suavizado.
- `numpy`: para construir ruidos controlados.
- `matplotlib.pyplot`: para comparar resultados.
- `pathlib.Path`: para cargar la imagen de trabajo.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

ruta_imagen = Path("Imagenes") / "lena.jpg"
imagen_bgr = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
if imagen_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")

imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
imagen_gris = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)


## 1. Imagen de referencia

Primero miramos la imagen original. Después vamos a degradarla de manera controlada para comparar cómo responde cada filtro.


In [ ]:
plt.figure(figsize=(7, 7), constrained_layout=True)
plt.imshow(imagen_rgb)
plt.title("Imagen original", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


## 2. Crear dos tipos de ruido

Vamos a generar dos degradaciones sintéticas:

- ruido gaussiano, que altera suavemente muchos píxeles;
- ruido sal y pimienta, que introduce puntos muy claros y muy oscuros.


In [ ]:
np.random.seed(42)

ruido_gaussiano = np.random.normal(loc=0, scale=18, size=imagen_gris.shape)
imagen_con_ruido_gaussiano = np.clip(imagen_gris.astype(np.float32) + ruido_gaussiano, 0, 255).astype(np.uint8)

imagen_con_sal_pimienta = imagen_gris.copy()
cantidad_pixeles = imagen_gris.size
cantidad_ruido = int(0.04 * cantidad_pixeles)

filas_sal = np.random.randint(0, imagen_gris.shape[0], cantidad_ruido)
columnas_sal = np.random.randint(0, imagen_gris.shape[1], cantidad_ruido)
imagen_con_sal_pimienta[filas_sal, columnas_sal] = 255

filas_pimienta = np.random.randint(0, imagen_gris.shape[0], cantidad_ruido)
columnas_pimienta = np.random.randint(0, imagen_gris.shape[1], cantidad_ruido)
imagen_con_sal_pimienta[filas_pimienta, columnas_pimienta] = 0


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
ejes[0].imshow(imagen_gris, cmap="gray")
ejes[0].set_title("Original en grises", fontweight="bold", loc="left")
ejes[0].axis("off")
ejes[1].imshow(imagen_con_ruido_gaussiano, cmap="gray")
ejes[1].set_title("Ruido gaussiano", fontweight="bold", loc="left")
ejes[1].axis("off")
ejes[2].imshow(imagen_con_sal_pimienta, cmap="gray")
ejes[2].set_title("Ruido sal y pimienta", fontweight="bold", loc="left")
ejes[2].axis("off")
plt.show()


## 3. Aplicar filtros de suavizado

Ahora vamos a comparar cuatro filtros comunes. No todos responden igual frente a cada degradación.


In [ ]:
promedio_gaussiano = cv2.blur(imagen_con_ruido_gaussiano, (5, 5))
gaussiano_gaussiano = cv2.GaussianBlur(imagen_con_ruido_gaussiano, (5, 5), 0)
mediana_gaussiano = cv2.medianBlur(imagen_con_ruido_gaussiano, 5)
bilateral_gaussiano = cv2.bilateralFilter(imagen_con_ruido_gaussiano, 9, 50, 50)

promedio_sal = cv2.blur(imagen_con_sal_pimienta, (5, 5))
gaussiano_sal = cv2.GaussianBlur(imagen_con_sal_pimienta, (5, 5), 0)
mediana_sal = cv2.medianBlur(imagen_con_sal_pimienta, 5)
bilateral_sal = cv2.bilateralFilter(imagen_con_sal_pimienta, 9, 50, 50)


In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(18, 8), constrained_layout=True)

titulos = [
    "Imagen con ruido",
    "Promedio",
    "Gaussiano",
    "Mediana",
    "Bilateral",
]

imagenes_fila_1 = [
    imagen_con_ruido_gaussiano,
    promedio_gaussiano,
    gaussiano_gaussiano,
    mediana_gaussiano,
    bilateral_gaussiano,
]

imagenes_fila_2 = [
    imagen_con_sal_pimienta,
    promedio_sal,
    gaussiano_sal,
    mediana_sal,
    bilateral_sal,
]

for indice, imagen in enumerate(imagenes_fila_1):
    ejes[0, indice].imshow(imagen, cmap="gray")
    ejes[0, indice].set_title(titulos[indice], fontweight="bold", loc="left")
    ejes[0, indice].axis("off")

for indice, imagen in enumerate(imagenes_fila_2):
    ejes[1, indice].imshow(imagen, cmap="gray")
    ejes[1, indice].set_title(titulos[indice], fontweight="bold", loc="left")
    ejes[1, indice].axis("off")

plt.show()


Compará por filas. En general, el filtro gaussiano suele responder bien al ruido gaussiano, mientras que la mediana suele funcionar mejor frente al ruido sal y pimienta. El bilateral intenta suavizar sin borrar tanto los bordes, pero no siempre resulta el más simple de interpretar.


## 4. Mirar un recorte pequeño

Un recorte ayuda a detectar mejor qué filtro conserva borde, qué filtro emborrona y qué filtro elimina mejor el ruido puntual.


In [ ]:
fila_inicial, fila_final = 190, 320
columna_inicial, columna_final = 190, 320

recorte_original = imagen_gris[fila_inicial:fila_final, columna_inicial:columna_final]
recorte_ruido = imagen_con_sal_pimienta[fila_inicial:fila_final, columna_inicial:columna_final]
recorte_mediana = mediana_sal[fila_inicial:fila_final, columna_inicial:columna_final]
recorte_bilateral = bilateral_sal[fila_inicial:fila_final, columna_inicial:columna_final]

fig, ejes = plt.subplots(1, 4, figsize=(14, 4), constrained_layout=True)
recortes = [recorte_original, recorte_ruido, recorte_mediana, recorte_bilateral]
titulos = ["Original", "Ruido", "Mediana", "Bilateral"]

for eje, imagen, titulo in zip(ejes, recortes, titulos):
    eje.imshow(imagen, cmap="gray")
    eje.set_title(titulo, fontweight="bold", loc="left")
    eje.axis("off")
plt.show()


## Actividad breve

Elegí uno de los dos tipos de ruido y escribí una comparación breve entre dos filtros. Indicá:

1. cuál elimina mejor el ruido;
2. cuál conserva mejor el borde;
3. cuál usarías si después tuvieras que detectar contornos.


## Cierre

Suavizar no significa simplemente “mejorar”. Cada filtro reduce algunas variaciones y al mismo tiempo borra parte del detalle. La pregunta útil no es qué filtro es más fuerte, sino qué filtro deja la imagen mejor preparada para la tarea que sigue.
